<a href="https://colab.research.google.com/github/Octavio1200/Finanzas-Computacionales/blob/main/Copia_de_Copia_de_VaRParam5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cálculo del VaR paramétrico de un portafolio formado por dos activos**

## **1. Carga librerías**

In [ ]:
##

## **2. Descarga los datos de Yahoo finance**

In [ ]:
## Descargar los datos de yfinance

In [ ]:
## Dataframe de los activos


In [ ]:
# Parte inicial del dataframe
stocks.head()

In [ ]:
# Parte final del dataframe
stocks.tail()

In [ ]:
# Algunas estadísticas
stocks.describe()

## **3. Calcular rendimientos, algunas estadísticas y gráficas.**

In [ ]:
## Rendimientos

In [ ]:
# Graficamos los rendimientos de los activos
for ticker in rendi.columns:
    plt.figure(figsize=(10, 6))
    plt.plot(rendi.index, rendi[ticker])
    plt.title(f'Rendimientos de {ticker}', fontsize=16)
    plt.xlabel('Fecha')
    plt.ylabel('Rendimiento')
    plt.grid(True, ls='--')
    plt.show()

In [ ]:
# Graficamos los rendimientos de ambas series
plt.figure(figsize=(10, 6))
plt.plot(rendi.index, rendi['ALSEA.MX'], label='ALSEA.MX')
plt.plot(rendi.index, rendi['CEMEXCPO.MX'], label='CEMEXCPO.MX')
plt.title('Rendimientos de ALSEA.MX y CEMEXCPO.MX', fontsize=16)
plt.xlabel('Fecha')
plt.ylabel('Rendimiento')
plt.legend()
plt.grid(True, ls='--')
plt.show()

In [ ]:
## Estadísticas de rendi: rendimientos

In [ ]:
## Media de los rendimientos

In [ ]:
# Calculamos media y desv. estandard de cada serie de rendimientos
rS1=np.array(rendi)[:,[0]];
rS2=np.array(rendi)[:,[1]];
mediaP1=[rS1.mean(), rS2.mean()]# Media muestral de cada activo
stdev=[np.std(rS1, ddof=1),np.std(rS2, ddof=1)]# Estimador insesgado de la varianza
print(f'La media de los activos = {mediaP1[0], mediaP1[1]}')
print(f'La desviación estándard de los activos= {stdev[0], stdev[1]}')
print("La media del primer activo: ${:.6f}".format(float(mediaP1[0])))
print("La media del segundo activo: ${:.6f}".format(float(mediaP1[1])))

In [ ]:
# Histograma del activo 1 y función de densidad normal ajustada por MV
plt.figure(num=1, figsize=(10, 7));#designa una figura donde se mostrará la gráfica
plt.hist(rS1, 50,density=True, color='w', edgecolor='blue', linewidth=2);#graficara solo la 1a. columna con 50 intervalos
plt.title('''Histograma de los rendimientos de ALSEA.MX''', fontsize=16)
#
overlay1 = np.linspace(rS1.min(), rS1.max(), 100)#sobrepone a la grafica anterior la gráfica de una Normal
mean1, std1 = norm.fit(rS1)
pdf1 = norm.pdf(overlay1, mean1, std1)
#plt.xlim(-9,6)
plt.plot(overlay1, pdf1, 'r-', linewidth=4);
plt.show()

In [ ]:
# Histograma del activo 2 y función de densidad normal ajustada por MV
plt.figure(num=1, figsize=(10, 7));#Gráfica del segundp activo
plt.hist(rS2, 50,density=True, color='white', edgecolor='orange', linewidth=2);
plt.title('''Histograma de los rendimientos de CEMEXCPO.MX''', fontsize=16)
#
overlay2 = np.linspace(rS2.min(), rS2.max(), 100)
mean2, std2 = norm.fit(rS2)
pdf2 = norm.pdf(overlay2, mean2, std2)
plt.plot(overlay2, pdf2, 'r-', linewidth=4);
plt.show()

In [ ]:
# Prueba de normalidad
import pylab
jarque_bera_test_rS1,  p_value = stats.jarque_bera(rS1)
print('jarque_bera_test_rActivo1=%.4f, p=%.6f' % (jarque_bera_test_rS1, p_value))
if p_value > 0.05:
    print('Probablemente Normal.')
else:
    print('Probablemente no Normal.')

In [ ]:
## Prueba visual con gráfica quantil-quantil

In [ ]:
# Establecemos posición en cada activo y del portafolio
# Calculamos la media ponderada
Posicion = [10000, 10000]
S_0 = stocks.values[-1, :]   # Últimos valores
P_0 = np.sum(Posicion * S_0) # Posición en u.m. del portafolio
P_00=Posicion*S_0 # Posición en cada activo
pesos_port=P_00/P_0 # Posiciones ponderadas
mediaP2=mediaP1*(pesos_port) # Promedio ponderado de cada activo
mediaP3=np.dot(mediaP1, pesos_port) # Promedio ponderado del portafolio
print(S_0)
print(P_0)
print(P_00)
print(pesos_port)
print(mediaP2)
print(mediaP3)

In [ ]:
## Matriz de varianza-covarianza

## **4. Cálculo del VaR con el Modelo de riskmetrics**
$$
\begin{equation}
\mathrm{VaR}_\alpha =\left( \mu  + {\Phi ^{ - 1}}(\alpha ){\sigma _\mathrm{P}}\sqrt T\right)\times M
\end{equation}
$$
donde:
- $\mu$: promedio de los rendimientos del portafolio.

- $\alpha$: nivel de confianza

- $\sigma _\mathrm{P}$:  desviación estándar del portafolio.

- ${\sigma _\mathrm{P}} = \sqrt {{\omega ^\prime}\Sigma \omega } $,

- $\omega: $ posiciones ponderadas del portafolio.

- $T$: horizonte de tiempo.

- $M$: valor de la posición del portafolio en unidades monetarias.

$$
\mathrm{VaR}_\alpha= \left[ { \mu + {\Phi ^{ - 1}}(\alpha )\times\sqrt{\begin{bmatrix}  {{\omega_1}{\omega_2} \cdots {\omega_n}} \\  \end{bmatrix}
\begin{bmatrix}
 \sigma_1^2 &\sigma_{12} & \cdots & \sigma_{1n}\\
 {{\sigma _{21}}}&{\sigma _2^2}& \cdots &{{\sigma _{2n}}}\\
\vdots & \vdots & \ddots & \vdots \\
 {{\sigma _{n1}}}&{{\sigma _{n2}}}& \cdots &{\sigma _n^2}
 \end{bmatrix}
\begin{bmatrix}
 {\omega _1} \\
 {\omega _2}  \\
  \vdots \\
  {\omega_n}
 \end{bmatrix}
 } \times \sqrt{T} } \right]\times M
$$


In [ ]:
# Función para calcular VaR por varianza-covarianza
def var_cov_VaR():
    Nivel_Conf = 0.99
    P_0 = np.sum(Posicion * S_0)
    pesos_port=P_00/P_0
    port_std = np.sqrt(pesos_port.T.dot(cov_var_matrix).dot(pesos_port))
    VaRP =  P_0*(mediaP3+port_std*norm.ppf(1-Nivel_Conf))
    return VaRP

In [ ]:
## Calculamos VaR del portafolio

* **EL VaR DEL PORTAFOLIO ES DE \$26,545.20, POR LO QUE  SE ESPERA QUE EL  PORTAFOLIO PERDERÁ \$26,545.20 Ó MENOS EN UN 99\% DE LOS ESCENARIOS, O BIEN SE ESPERA PERDERÁ $26,545.20 Ó MÁS EN EL 1\% DE LOS ESCENARIOS.**

In [ ]:
# Sigma_P Volatilidad del portafolio (desviación estándard)ñ
# Es un escalar.
port_std = np.sqrt(pesos_port.T.dot(cov_var_matrix).dot(pesos_port))
port_std

In [ ]:
# VaR paramétrico del primer activo
Nivel_Conf = 0.99
VaRActivo1=(mediaP1[0]+norm.ppf(1-Nivel_Conf)*stdev[0])*P_00[0]
print("El Valor en Riesgo al 99% de confianza del Activo 1: ${:.6f}".format(float(VaRActivo1)))

In [ ]:
## VaR paramétrico del segundo activo

## **5. Cálculo del VaR por definición de la inversa de la función de distribución**

In [ ]:
# VaR del portafolio por definición de la inversa de la función de distribución.
MediaP=np.dot(mediaP1,P_00)
print("La media ponderada del portafolio es: ${:.6f}".format(float(MediaP)))
Port_stdP = np.sqrt(P_00.T.dot(cov_var_matrix).dot(P_00))
print("La desviación estándard del portafolio es: ${:.6f}".format(float(Port_stdP)))
VaRP2=norm.ppf(1-Nivel_Conf,MediaP,Port_stdP)
print("El Valor en Riesgo al 99% de confianza es: ${:.6f}".format(float(VaRP2)))


In [ ]:
# VaR por definición del primer activo
MediaPA1=mediaP1[0]*P_00[0]
print(MediaPA1)
DesvEstPA1=stdev[0]*P_00[0]
print(DesvEstPA1)
VaRPA1=norm.ppf(1-Nivel_Conf,MediaPA1,DesvEstPA1)
print("El Valor en Riesgo al 99% de confianza del Activo 1: ${:.6f}".format(float(VaRPA1)))

In [ ]:
# VaR por definición del segundo activo ###
MediaPA2=mediaP1[1]*P_00[1]
print(MediaPA2)
DesvEstPA2=stdev[1]*P_00[1]
print(DesvEstPA2)
VaRPA2=norm.ppf(1-Nivel_Conf,MediaPA2,DesvEstPA2)
print("El Valor en Riesgo al 99% de confianza del Activo 2: ${:.6f}".format(float(VaRPA2)))

- **Subaditividad**. Para toda $L_1 + L_2 \in \mathcal{M}$ tenemos que $\varrho(L_1+L_2) \leq \varrho(L_1)+ \varrho(L_2)$. La idea principal de este propiedad es que "una unión no crea riesgo extra", esto refleja la idea de que el riesgo se puede reducir diversificando los activos.

In [ ]:
# Comparamos el VaR del portafolio versus la suma de los VaR's de cada activo
print("El Valor en Riesgo al 99% de confianza del portafolio es: ${:.6f}".format(float(VaRP2)))
print("La suma del VaR del activo 1 y del activo 2 es: ${:.6f}".format(float(VaRPA1+ VaRPA2)))

## **6. Conclusión:**

- **Calcular el VaR por el método de varianza-covariaza (RiskMetrics) y por la inversa de la función de distribución, proporcionan resultados iguales.**

- **¿Qué otra distribución podemos usar?.**